# 04 - Export para Clinical Eval App

Este notebook prepara el paquete de evaluacion para `clinical_eval_app/`:
- Lee `results.jsonl` de un escenario (A/B/C/D/E).
- Filtra casos correctos (`class_match=True`) si se desea.
- Ordena de mejor a peor por `iou_score + proximity_score`.
- Genera imagenes con **solo Ground Truth bbox** dibujado en verde.
- Exporta `clinical_eval_app/data/cases.json` y metadatos auxiliares.

In [11]:
from __future__ import annotations

import csv
import json
from pathlib import Path
from typing import Any

import cv2
import numpy as np

try:
    from PIL import Image
except Exception:
    Image = None

print('OK: imports cargados')

OK: imports cargados


## Paso 1 - Configuracion del export
En la siguiente celda defines escenario, run, cantidad de casos y politica de reemplazo.
Si REPLACE_EXISTING_EXPORT = True, el notebook limpia imagenes previas de la app y sobrescribe los metadatos de salida.

In [12]:
# =========================
# Configuracion del export
# =========================
SCENARIO_NAME = 'scenario_B'      # scenario_A .. scenario_E
RUN_ID = None                      # ejemplo: '20260407_103355' (None => ultimo run)
RUN_PATH_OVERRIDE = None           # Path absoluto/relativo a run_... si quieres forzar

TOP_K = 50                         # cuantas imagenes exportar
ONLY_CLASS_MATCH = True            # solo muestras correctas
REQUIRE_STATUS_OK = True           # exige status == 'ok'
DROP_EMPTY_JUSTIFICATION = True    # descarta si falta clinical_justification

# Score de ranking solicitado: iou + proximity
# (si falta alguno, se trata como 0.0)

# Salida para la web app
OUTPUT_APP_DIR = Path('clinical_eval_app')
OUTPUT_DATA_DIR = OUTPUT_APP_DIR / 'data'
OUTPUT_IMAGES_DIR = OUTPUT_DATA_DIR / 'images'
OUTPUT_CASES_JSON = OUTPUT_DATA_DIR / 'cases.json'
OUTPUT_EXPORT_CSV = OUTPUT_DATA_DIR / 'cases_export_debug.csv'

# Si True, borra imagenes previas exportadas y reemplaza metadatos actuales
REPLACE_EXISTING_EXPORT = True

# Dibujo
GT_COLOR_BGR = (0, 200, 0)
GT_THICKNESS = 3
SCALE_MAX = 1000

print('Configuracion lista')

Configuracion lista


## Paso 2 - Resolucion de rutas y utilidades
La celda siguiente prepara funciones auxiliares para localizar el proyecto, resolver el run seleccionado y convertir bboxes.
Tambien deja definida la ruta final de `results.jsonl` a procesar.

In [13]:
def detect_project_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    candidates = [base, *base.parents]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'data').exists():
            return candidate
    return base


def resolve_run_dir(project_root: Path) -> Path:
    if RUN_PATH_OVERRIDE:
        raw = Path(RUN_PATH_OVERRIDE)
        run_dir = raw if raw.is_absolute() else (project_root / raw)
        run_dir = run_dir.resolve()
        if not (run_dir / 'results.jsonl').exists():
            raise FileNotFoundError(f'No existe results.jsonl en: {run_dir}')
        return run_dir

    scenario_dir = project_root / 'data' / 'processed' / 'scenario_results' / SCENARIO_NAME
    if not scenario_dir.exists():
        raise FileNotFoundError(f'Escenario no encontrado: {scenario_dir}')

    if RUN_ID:
        run_dir = scenario_dir / f'run_{RUN_ID}'
        if not (run_dir / 'results.jsonl').exists():
            raise FileNotFoundError(f'No existe results.jsonl para run_id={RUN_ID}: {run_dir}')
        return run_dir

    runs = sorted(
        [p for p in scenario_dir.glob('run_*') if (p / 'results.jsonl').exists()],
        key=lambda p: p.name,
    )
    if not runs:
        raise FileNotFoundError(f'No hay runs con results.jsonl en: {scenario_dir}')
    return runs[-1]


def read_jsonl_records(jsonl_path: Path) -> list[dict[str, Any]]:
    records: list[dict[str, Any]] = []
    with jsonl_path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            if '__scenario_meta__' in obj:
                continue
            if isinstance(obj, dict):
                records.append(obj)
    return records


def to_float_or_none(value: Any) -> float | None:
    try:
        if value is None:
            return None
        return float(value)
    except Exception:
        return None


def to_bbox_int4(value: Any) -> list[int] | None:
    if not isinstance(value, (list, tuple)) or len(value) != 4:
        return None
    out: list[int] = []
    for item in value:
        try:
            out.append(int(round(float(item))))
        except Exception:
            return None
    return out


def score_iou_plus_proximity(rec: dict[str, Any]) -> float:
    iou = to_float_or_none(rec.get('iou_score')) or 0.0
    prox = to_float_or_none(rec.get('proximity_score')) or 0.0
    return iou + prox


def load_image_bgr(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is not None:
        return img

    if Image is not None:
        pil = Image.open(path).convert('RGB')
        return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

    raise RuntimeError(f'No se pudo cargar imagen: {path}')


def denormalize_bbox_xyxy(bbox_norm: list[int], width: int, height: int) -> tuple[int, int, int, int]:
    # Formato esperado: [xmin, ymin, xmax, ymax] en escala 0..1000
    xmin_n, ymin_n, xmax_n, ymax_n = bbox_norm

    xmin = int((xmin_n / SCALE_MAX) * width)
    ymin = int((ymin_n / SCALE_MAX) * height)
    xmax = int((xmax_n / SCALE_MAX) * width)
    ymax = int((ymax_n / SCALE_MAX) * height)

    xmin = max(0, min(xmin, width - 1))
    xmax = max(0, min(xmax, width - 1))
    ymin = max(0, min(ymin, height - 1))
    ymax = max(0, min(ymax, height - 1))

    return xmin, ymin, xmax, ymax


def draw_gt_bbox_only(image_bgr: np.ndarray, gt_bbox_norm: list[int]) -> np.ndarray:
    out = image_bgr.copy()
    h, w = out.shape[:2]
    x1, y1, x2, y2 = denormalize_bbox_xyxy(gt_bbox_norm, w, h)
    cv2.rectangle(out, (x1, y1), (x2, y2), GT_COLOR_BGR, GT_THICKNESS)
    return out


PROJECT_ROOT = detect_project_root()
RUN_DIR = resolve_run_dir(PROJECT_ROOT)
RESULTS_JSONL = RUN_DIR / 'results.jsonl'

# Asegura que la salida de la app siempre quede en la raiz del proyecto,
# incluso si el notebook se ejecuta desde la carpeta notebooks/.
OUTPUT_APP_DIR = OUTPUT_APP_DIR if OUTPUT_APP_DIR.is_absolute() else (PROJECT_ROOT / OUTPUT_APP_DIR)
OUTPUT_APP_DIR = OUTPUT_APP_DIR.resolve()
OUTPUT_DATA_DIR = OUTPUT_APP_DIR / 'data'
OUTPUT_IMAGES_DIR = OUTPUT_DATA_DIR / 'images'
OUTPUT_CASES_JSON = OUTPUT_DATA_DIR / 'cases.json'
OUTPUT_EXPORT_CSV = OUTPUT_DATA_DIR / 'cases_export_debug.csv'

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'RUN_DIR:      {RUN_DIR}')
print(f'JSONL:        {RESULTS_JSONL}')
print(f'APP_DIR:      {OUTPUT_APP_DIR}')
print(f'DATA_DIR:     {OUTPUT_DATA_DIR}')

PROJECT_ROOT: C:\Users\david\Desktop\TFG\TFG_VLM_Medical
RUN_DIR:      C:\Users\david\Desktop\TFG\TFG_VLM_Medical\data\processed\scenario_results\scenario_B\run_20260416_155934
JSONL:        C:\Users\david\Desktop\TFG\TFG_VLM_Medical\data\processed\scenario_results\scenario_B\run_20260416_155934\results.jsonl
APP_DIR:      C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app
DATA_DIR:     C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data


## Paso 3 - Filtro y ranking de casos
Aqui se aplica el filtrado (status, class_match, justificacion, etc.) y se ordena por `iou_score + proximity_score` de mayor a menor.
El resultado de esta fase es la lista `selected` que se exportara a la app.

In [14]:
records = read_jsonl_records(RESULTS_JSONL)
print(f'Registros leidos: {len(records)}')

prepared: list[dict[str, Any]] = []
for rec in records:
    if REQUIRE_STATUS_OK and str(rec.get('status', '')).lower() != 'ok':
        continue

    if ONLY_CLASS_MATCH and not bool(rec.get('class_match')):
        continue

    gt_bbox = to_bbox_int4(rec.get('ground_truth_bbox'))
    if gt_bbox is None:
        continue

    payload = rec.get('payload') if isinstance(rec.get('payload'), dict) else {}
    justification = str(payload.get('clinical_justification') or '').strip()
    if DROP_EMPTY_JUSTIFICATION and not justification:
        continue

    image_path_raw = str(rec.get('image_path') or '').strip()
    if not image_path_raw:
        continue

    image_path = Path(image_path_raw)
    image_path = image_path if image_path.is_absolute() else (PROJECT_ROOT / image_path)
    image_path = image_path.resolve()
    if not image_path.exists():
        continue

    iou = to_float_or_none(rec.get('iou_score'))
    prox = to_float_or_none(rec.get('proximity_score'))
    score = (iou or 0.0) + (prox or 0.0)

    prepared.append({
        'image_id': rec.get('image_id'),
        'image_name': str(rec.get('image_name') or image_path.name),
        'image_path': image_path,
        'ground_truth_cls': str(rec.get('ground_truth_cls') or '').strip(),
        'ground_truth_bbox': gt_bbox,
        'clinical_justification': justification,
        'id_modelo': str(rec.get('model_id') or '').strip(),
        'iou_score': iou,
        'proximity_score': prox,
        'quality_score': score,
    })

prepared.sort(key=lambda x: x['quality_score'], reverse=True)
selected = prepared[: max(0, int(TOP_K))]

print(f'Candidatos tras filtros: {len(prepared)}')
print(f'Seleccion final (TOP_K): {len(selected)}')

for i, row in enumerate(selected[:5], start=1):
    print(f"{i:02d}. image_id={row['image_id']} score={row['quality_score']:.4f} iou={row['iou_score']} prox={row['proximity_score']} cls={row['ground_truth_cls']}")

Registros leidos: 501
Candidatos tras filtros: 494
Seleccion final (TOP_K): 50
01. image_id=741 score=1.9420 iou=0.9589588446829157 prox=0.9829983643056763 cls=ASS
02. image_id=422 score=1.8479 iou=0.8949108683151237 prox=0.9529772941503185 cls=AD
03. image_id=563 score=1.8468 iou=0.8911969696969697 prox=0.9555835097730034 cls=AD
04. image_id=779 score=1.8233 iou=0.8741703005722766 prox=0.9491477813605984 cls=AD
05. image_id=462 score=1.8189 iou=0.8722219064400614 prox=0.946713227174648 cls=AD


## Paso 4 - Export y reemplazo de datos de la app
Esta celda crea `clinical_eval_app/data` si no existe.
Si el reemplazo esta activo, elimina imagenes previas en `clinical_eval_app/data/images` y luego escribe de nuevo `cases.json` y el CSV auxiliar.

In [15]:
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CASES_JSON.parent.mkdir(parents=True, exist_ok=True)

# Reemplazo completo del export anterior: limpia imagenes viejas del paquete web
if REPLACE_EXISTING_EXPORT:
    for img_file in OUTPUT_IMAGES_DIR.glob('*'):
        if img_file.is_file():
            img_file.unlink()

cases: list[dict[str, Any]] = []
csv_rows: list[dict[str, Any]] = []

for idx, row in enumerate(selected, start=1):
    src_img_path: Path = row['image_path']
    gt_bbox_norm: list[int] = row['ground_truth_bbox']

    image_bgr = load_image_bgr(src_img_path)
    image_annotated = draw_gt_bbox_only(image_bgr, gt_bbox_norm)

    out_name = f"case_{idx:03d}_{src_img_path.stem}_gtbbox.jpg"
    out_abs = OUTPUT_IMAGES_DIR / out_name
    ok = cv2.imwrite(str(out_abs), image_annotated)
    if not ok:
        raise RuntimeError(f'No se pudo guardar: {out_abs}')

    id_imagen = str(src_img_path.stem)
    image_file_for_app = f"data/images/{out_name}"

    case_obj = {
        'id_imagen': id_imagen,
        'image_path': image_file_for_app,
        'ground_truth_class': row['ground_truth_cls'],
        'id_modelo': row['id_modelo'],
        'clinical_justification': row['clinical_justification'],
    }
    cases.append(case_obj)

    csv_rows.append({
        'id_imagen': id_imagen,
        'id_modelo': row['id_modelo'],
        'veredicto': '',
        'comentario_medico': '',
        'source_image_id': row['image_id'],
        'source_image_name': row['image_name'],
        'source_image_path': str(src_img_path),
        'exported_image_path': image_file_for_app,
        'true_class': row['ground_truth_cls'],
        'iou_score': row['iou_score'],
        'proximity_score': row['proximity_score'],
        'quality_score': row['quality_score'],
    })

# Sobrescribe siempre cases.json, incluso si queda vacio
with OUTPUT_CASES_JSON.open('w', encoding='utf-8') as f:
    json.dump(cases, f, ensure_ascii=False, indent=2)

with OUTPUT_EXPORT_CSV.open('w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(csv_rows[0].keys()) if csv_rows else [
        'id_imagen', 'id_modelo', 'veredicto', 'comentario_medico',
        'source_image_id', 'source_image_name', 'source_image_path', 'exported_image_path',
        'true_class', 'iou_score', 'proximity_score', 'quality_score'
    ])
    writer.writeheader()
    writer.writerows(csv_rows)

print(f'Export completado. Cases: {len(cases)}')
print(f'JSON: {OUTPUT_CASES_JSON.resolve()}')
print(f'CSV:  {OUTPUT_EXPORT_CSV.resolve()}')
print(f'IMG:  {OUTPUT_IMAGES_DIR.resolve()}')
print(f'Reemplazo activo: {REPLACE_EXISTING_EXPORT}')

Export completado. Cases: 50
JSON: C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data\cases.json
CSV:  C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data\cases_export_debug.csv
IMG:  C:\Users\david\Desktop\TFG\TFG_VLM_Medical\clinical_eval_app\data\images
Reemplazo activo: True


## Paso 5 - Verificacion rapida
Se imprime una muestra de los primeros casos para comprobar formato y rutas resultantes.
Si el total de casos es 0, revisa filtros como `ONLY_CLASS_MATCH` o la disponibilidad de justificaciones.

In [16]:
# Vista rapida de salida
print('Primeros 3 casos exportados:')
for item in cases[:3]:
    print(json.dumps(item, ensure_ascii=False, indent=2))

print('...')
print(f'Total casos exportados: {len(cases)}')

Primeros 3 casos exportados:
{
  "id_imagen": "741",
  "image_path": "data/images/case_001_741_gtbbox.jpg",
  "ground_truth_class": "ASS",
  "id_modelo": "qwen3_5-9b@q8_0",
  "clinical_justification": "La lesión presenta una morfología plana a ligeramente protrusiva con bordes irregulares y un aspecto de 'cresta' o proyección hacia la luz intestinal. La superficie muestra una textura rugosa y desorganizada, con áreas de coloración amarillenta que sugieren depósitos de moco o cambios epiteliales. El patrón vascular es difuso y poco definido en comparación con el tejido circundante sano, lo cual es característico de las lesiones serradas."
}
{
  "id_imagen": "422",
  "image_path": "data/images/case_002_422_gtbbox.jpg",
  "ground_truth_class": "AD",
  "id_modelo": "qwen3_5-9b@q8_0",
  "clinical_justification": "La imagen muestra una lesión prominente y bien definida que se eleva claramente sobre el nivel de la mucosa circundante. Su superficie es lisa pero presenta un patrón vascular suti